In [ ]:
"""
Multi-Modal Brain Time-Series Extraction Pipeline (fMRI & MEG)
Features:
- AAL3 Atlas Network Assignment & Robust ROI Padding (NaN handling)
- MEG Source Reconstruction (LCMV Beamformer) across 8 frequency bands
- FOOOF + VMD (Variational Mode Decomposition) for Aperiodic Time-Series Extraction
"""

import os
import glob
import zipfile
import tempfile
import re
import gc
import logging
import warnings
import numpy as np
import pandas as pd
import nibabel as nib
from scipy.linalg import sqrtm
from scipy import signal
import mne
from mne.beamformer import make_lcmv, apply_lcmv_epochs
from nilearn.maskers import NiftiLabelsMasker
from nilearn import plotting
from fooof import FOOOF, FOOOFGroup
from vmdpy import VMD
from joblib import Parallel, delayed

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# ==============================================================================
# 1. Configuration & Directories
# ==============================================================================
AAL3_DIR = r'E:\AAL3_Atlas'
MEG_INPUT_DIR = r'E:\HCP_MEG_preprocessing'
MEG_OUTPUT_DIR = r'E:\HCP_MEG_Timeseries_AAL3_Full'
FMRI_INPUT_DIR = r'E:\HCP\fMRI_preproc'
FMRI_OUTPUT_DIR = r'E:\HCP\HCP_fMRI_Timeseries_AAL3_Full'
APERIODIC_OUTPUT_DIR = MEG_OUTPUT_DIR  # Save aperiodic timeseries in MEG directory

AAL3_NII_PATH = os.path.join(AAL3_DIR, 'AAL3v1.nii')
AAL3_TXT_PATH = os.path.join(AAL3_DIR, 'AAL3v1.nii.txt')

FREQ_BANDS = {
    'Low-Frequency': (0.1, 1.5), 'Delta': (1.5, 4.0), 'Theta': (4.0, 8.0),
    'Alpha': (8.0, 13.0), 'Beta': (13.0, 30.0), 'Low-Gamma': (30.0, 55.0),
    'High-Gamma': (65.0, 120.0), 'Wide-band': (0.1, 120.0)
}

MEG_SFREQ = 250.0  # Resampling frequency for MEG

# ==============================================================================
# 2. Atlas Parsing & Network Assignment
# ==============================================================================
def assign_network(roi_name):
    """Map AAL3 regions to macro-scale functional networks."""
    name = roi_name.lower()
    if any(x in name for x in ['cingulate_post', 'precuneus', 'angular', 'frontal_med_orb', 'frontal_sup_medial', 'rectus']):
        return 'Default_Mode_Network'
    elif any(x in name for x in ['insula', 'cingulate_ant', 'cingulate_mid', 'acc_', 'supramarginal', 'frontal_inf_oper']):
        return 'Salience_Network'
    elif any(x in name for x in ['frontal_sup', 'frontal_mid', 'parietal_inf', 'parietal_sup', 'frontal_inf_tri']):
        return 'Executive_Control_Network'
    elif any(x in name for x in ['occipital', 'calcarine', 'cuneus', 'lingual', 'fusiform']):
        return 'Visual_Network'
    elif any(x in name for x in ['precentral', 'postcentral', 'supp_motor', 'paracentral', 'rolandic']):
        return 'Sensorimotor_Network'
    elif any(x in name for x in ['temporal_sup', 'heschl']):
        return 'Auditory_Network'
    elif any(x in name for x in ['hippocampus', 'parahippocampal', 'amygdala', 'caudate', 'putamen', 'pallidum', 'n_acc', 'olfactory', 'temporal_pole', 'ofc']):
        return 'Limbic_Subcortical_System'
    elif 'thal' in name or name == 'thalamus':
        return 'Thalamic_System'
    elif any(x in name for x in ['cerebelum', 'cerebellum', 'vermis']):
        return 'Cerebellar_System'
    elif any(x in name for x in ['vta', 'sn_', 'lc_', 'raphe', 'red_n']):
        return 'Brainstem'
    elif 'temporal' in name:
        return 'Temporal_Other'
    else:
        return 'Other'

def load_aal3_labels(txt_path):
    """Load ROI names, Network assignments, and integer IDs from AAL3 txt file."""
    labels, networks, roi_ids = [], [], []
    with open(txt_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                name = next((p for p in parts if not p.isdigit()), None)
                roi_id_str = next((p for p in parts if p.isdigit()), None)
                if name and roi_id_str:
                    labels.append(name)
                    networks.append(assign_network(name))
                    roi_ids.append(int(roi_id_str)) 
    return pd.DataFrame({'Network': networks, 'AAL_Label': labels, 'ROI_ID': roi_ids})

# ==============================================================================
# 3. MEG Feature Extraction (Beamforming & FOOOF)
# ==============================================================================
def symmetric_orthogonalization(timeseries):
    cov_mat = np.cov(timeseries)
    try:
        inv_sqrt_cov = np.real(sqrtm(np.linalg.inv(cov_mat)))
    except np.linalg.LinAlgError:
        inv_sqrt_cov = np.real(sqrtm(np.linalg.pinv(cov_mat)))
    return np.dot(inv_sqrt_cov, timeseries)

# Hotfix for MNE FieldTrip channel parsing
import mne.io.fieldtrip.utils
original_process_channel_meg = mne.io.fieldtrip.utils._process_channel_meg
def patched_process_channel_meg(cur_ch, grad):
    try:
        labels = [str(L) for L in grad["label"]]
        if cur_ch["ch_name"] in labels:
            idx = labels.index(cur_ch["ch_name"])
            if "chantype" in grad and grad["chantype"][idx] in ["meg", "mag"]:
                grad["chantype"][idx] = "megmag" 
    except Exception: pass
    return original_process_channel_meg(cur_ch, grad)
mne.io.fieldtrip.utils._process_channel_meg = patched_process_channel_meg

def parse_subject_transform(txt_path):
    with open(txt_path, 'r') as f: 
        content = f.read()
    match = re.search(r'transform\.bti2spm\s*=\s*\[(.*?)\];', content, re.DOTALL)
    if not match: 
        raise ValueError("Could not find bti2spm matrix")
    mat = [[float(x) for x in line.split() if x.strip()] for line in match.group(1).strip().split('\n') if line.strip()]
    bti2spm_mat = np.array(mat)
    bti2spm_mat[:3, 3] /= 1000.0 
    return mne.Transform('head', 'mri', bti2spm_mat)

def process_single_meg_file(mat_path, df_labels, roi_coords, base_name, trans_to_mni, subj):
    n_regions = len(df_labels)
    try:
        epochs = mne.read_epochs_fieldtrip(mat_path, info=None, data_name='data')
        epochs.load_data()
    except Exception: return

    epochs.resample(MEG_SFREQ)
    for ch in epochs.info['chs']: 
        ch['coord_frame'] = mne.io.constants.FIFF.FIFFV_COORD_HEAD
    
    fs_dir = mne.datasets.fetch_fsaverage(verbose=False)
    mri_file = os.path.join(fs_dir, 'mri', 'brain.mgz')
    bem_file = os.path.join(fs_dir, 'bem', 'fsaverage-5120-5120-5120-bem-sol.fif')
    src = mne.setup_volume_source_space('fsaverage', pos=5.0, mri=mri_file, bem=bem_file, subjects_dir=os.path.dirname(fs_dir), verbose=False)
    
    try:
        fwd = mne.make_forward_solution(epochs.info, trans=trans_to_mni, src=src, bem=bem_file, meg=True, eeg=False, verbose=False)
    except Exception: return
    
    inuse_vertno = src[0]['vertno'] 
    mne_src_coords = src[0]['rr'][inuse_vertno] * 1000.0 
    roi_mapping, valid_rois = {}, []
    
    for i, target_coord in enumerate(roi_coords):
        distances = np.linalg.norm(mne_src_coords - target_coord, axis=1)
        points_in_radius = np.where(distances <= 8.0)[0]
        roi_mapping[i] = points_in_radius
        if len(points_in_radius) > 0: 
            valid_rois.append(i)
            
    wide_band_roi_epochs = None 
    
    for band, (l_freq, h_freq) in FREQ_BANDS.items():
        out_parquet = os.path.join(MEG_OUTPUT_DIR, f"{base_name}_{band}.parquet")
        if os.path.exists(out_parquet): continue
            
        logging.info(f"      -> Extracting MEG Band: {band}...")
        ep_filt = epochs.copy().filter(l_freq, h_freq, fir_design='firwin', verbose=False)
        data_cov = mne.compute_covariance(ep_filt, method='empirical', verbose=False)
        filters = make_lcmv(ep_filt.info, fwd, data_cov, reg=0.05, pick_ori='max-power', weight_norm='unit-noise-gain', verbose=False)
        
        stcs = apply_lcmv_epochs(ep_filt, filters, return_generator=True, verbose=False)
        all_epochs_rois = []
        for stc in stcs:
            epoch_roi_ts = np.full((n_regions, stc.data.shape[1]), np.nan)
            for i in valid_rois:
                region_data = stc.data[roi_mapping[i], :]
                if region_data.shape[0] > 0:
                    u, s, vh = np.linalg.svd(region_data - np.mean(region_data, axis=1, keepdims=True), full_matrices=False)
                    epoch_roi_ts[i, :] = vh[0, :]
            all_epochs_rois.append(epoch_roi_ts)
            
        roi_ts_matrix = np.hstack(all_epochs_rois)
        if band == 'Wide-band': 
            wide_band_roi_epochs = all_epochs_rois 
        
        roi_ts_matrix[valid_rois, :] = symmetric_orthogonalization(roi_ts_matrix[valid_rois, :])
        
        index = pd.MultiIndex.from_arrays([df_labels['Network'], df_labels['AAL_Label']], names=['Network', 'Region'])
        df_out = pd.DataFrame(roi_ts_matrix.T, columns=index, dtype=np.float32)
        df_out.index.name = 'Timepoint'
        df_out.to_parquet(out_parquet, engine='pyarrow', compression='snappy')
        
        del ep_filt, stcs, all_epochs_rois, roi_ts_matrix, df_out
        gc.collect()

    # FOOOF Parameter Extraction
    if wide_band_roi_epochs is not None:
        out_fooof = os.path.join(MEG_OUTPUT_DIR, f"{base_name}_Aperiodic_Params.xlsx")
        if not os.path.exists(out_fooof):
            logging.info("      -> Computing FOOOF Aperiodic Features...")
            freqs = np.fft.rfftfreq(wide_band_roi_epochs[0].shape[1], 1/MEG_SFREQ)
            all_psds = [np.abs(np.fft.rfft(epoch_data[valid_rois, :], axis=-1))**2 for epoch_data in wide_band_roi_epochs]
            mean_psd = np.mean(all_psds, axis=0) 
            
            fg = FOOOFGroup(peak_width_limits=[1, 8], max_n_peaks=6, aperiodic_mode='knee', verbose=False)
            fg.fit(freqs, mean_psd, [1.0, 50.0]) 
            
            final_ap = np.full((n_regions, 4), np.nan)
            for counter, i in enumerate(valid_rois):
                final_ap[i, 0:3] = fg.get_params('aperiodic_params')[counter]
                final_ap[i, 3] = fg.get_params('r_squared')[counter]
                
            index = pd.MultiIndex.from_arrays([df_labels['Network'], df_labels['AAL_Label']], names=['Network', 'Region'])
            df_fooof = pd.DataFrame(final_ap, index=index, columns=['Offset', 'Knee', 'Exponent', 'R_squared'])
            df_fooof.to_excel(out_fooof)
            
            del all_psds, mean_psd, fg, final_ap, df_fooof
            gc.collect()

def run_meg_pipeline(roi_coords, df_labels):
    logging.info("="*50)
    logging.info(" PHASE 1: MEG Feature Extraction Pipeline")
    logging.info("="*50)
    os.makedirs(MEG_OUTPUT_DIR, exist_ok=True)
    anatomy_zips = glob.glob(os.path.join(MEG_INPUT_DIR, '*_MEG_anatomy.zip'))
    
    for anatomy_zip in anatomy_zips:
        subj = os.path.basename(anatomy_zip).split('_')[0]
        logging.info(f"[MEG] Processing Subject: {subj}")
        with tempfile.TemporaryDirectory() as temp_dir:
            try:
                with zipfile.ZipFile(anatomy_zip, 'r') as zf:
                    txt_file = [f for f in zf.namelist() if f.endswith('transform.txt')][0]
                    temp_txt_path = zf.extract(txt_file, temp_dir)
                    trans_to_mni = parse_subject_transform(temp_txt_path)
            except Exception: continue

            task_zips = glob.glob(os.path.join(MEG_INPUT_DIR, f"{subj}_MEG_*_preproc.zip"))
            for task_zip in task_zips:
                with zipfile.ZipFile(task_zip, 'r') as zf:
                    meg_files = [f for f in zf.namelist() if f.endswith('.mat') and ('rmegpreproc' in f or 'tmegpreproc' in f) and ('trialinfo' not in f)]
                    for meg_file_in_zip in meg_files:
                        base_name = os.path.basename(meg_file_in_zip).replace('.mat', '')
                        logging.info(f"   - Analyzing: {base_name}")
                        temp_mat_path = zf.extract(meg_file_in_zip, temp_dir)
                        process_single_meg_file(temp_mat_path, df_labels, roi_coords, base_name, trans_to_mni, subj)
                        os.remove(temp_mat_path)

# ==============================================================================
# 4. MEG Aperiodic Signal Separation (VMD)
# ==============================================================================
def process_single_vmd_chunk(chunk_data, sfreq):
    if not np.isfinite(chunk_data).all() or np.allclose(chunk_data, chunk_data[0]):
        return chunk_data 

    freqs, psd = signal.welch(chunk_data, fs=sfreq, nperseg=int(sfreq * 2))
    psd = np.clip(psd, a_min=1e-12, a_max=None)
    
    fm = FOOOF(peak_width_limits=[1, 8], max_n_peaks=4, aperiodic_mode='fixed', verbose=False)
    try:
        fm.fit(freqs, psd, [1.0, 50.0])
    except Exception:
        return chunk_data
        
    peaks = fm.get_params('peak_params')
    if peaks is None or len(peaks) == 0:
        return chunk_data

    K = len(peaks) + 2 
    try:
        u, u_hat, omega = VMD(chunk_data, alpha=2000, tau=0., K=K, DC=0, init=1, tol=1e-7)
    except Exception:
        return chunk_data

    aperiodic_signal = np.zeros_like(chunk_data)
    for k in range(K):
        imf = u[k, :]
        f_imf, psd_imf = signal.welch(imf, fs=sfreq, nperseg=int(sfreq))
        dominant_freq = f_imf[np.argmax(psd_imf)]
        
        is_periodic = False
        for peak in peaks:
            cf, _, bw = peak 
            if (cf - bw) <= dominant_freq <= (cf + bw):
                is_periodic = True
                break
                
        if not is_periodic:
            aperiodic_signal += imf

    return aperiodic_signal

def extract_aperiodic_for_region(region_ts, sfreq, chunk_len_sec=10):
    chunk_size = int(sfreq * chunk_len_sec)
    n_samples = len(region_ts)
    aperiodic_ts = np.zeros(n_samples)
    
    for start_idx in range(0, n_samples, chunk_size):
        end_idx = min(start_idx + chunk_size, n_samples)
        chunk = region_ts[start_idx:end_idx]
        
        if len(chunk) < sfreq:
            aperiodic_ts[start_idx:end_idx] = chunk
            continue
            
        aperiodic_ts[start_idx:end_idx] = process_single_vmd_chunk(chunk, sfreq)
        
    return aperiodic_ts

def run_vmd_pipeline():
    logging.info("="*50)
    logging.info(" PHASE 2: MEG VMD Aperiodic Signal Separation")
    logging.info("="*50)
    os.makedirs(APERIODIC_OUTPUT_DIR, exist_ok=True)
    wide_band_files = glob.glob(os.path.join(MEG_OUTPUT_DIR, '*_Wide-band.parquet'))
    
    if not wide_band_files:
        logging.warning(" No Wide-band.parquet files found for VMD processing.")
        return

    for parquet_path in wide_band_files:
        filename = os.path.basename(parquet_path)
        output_path = os.path.join(APERIODIC_OUTPUT_DIR, filename.replace('Wide-band', 'Aperiodic-VMD'))
        
        if os.path.exists(output_path):
            logging.info(f" [Skipped] {filename} (Aperiodic-VMD already exists)")
            continue

        logging.info(f" [VMD Processing] {filename} ...")
        try:
            df_wide = pd.read_parquet(parquet_path)
            results = Parallel(n_jobs=-1, backend="multiprocessing")(
                delayed(extract_aperiodic_for_region)(df_wide.iloc[:, col].values, MEG_SFREQ) 
                for col in range(df_wide.shape[1])
            )
            
            df_aperiodic = pd.DataFrame(np.column_stack(results), index=df_wide.index, columns=df_wide.columns)
            df_aperiodic.columns.names = df_wide.columns.names 
            df_aperiodic.index.name = df_wide.index.name
            
            df_aperiodic.to_parquet(output_path, engine='pyarrow', compression='snappy')
            logging.info(f" [Success] {filename} Aperiodic-VMD series generated.")
        except Exception as e:
            logging.error(f" [Failed] Error processing {filename}: {str(e)}")
        finally:
            gc.collect()

# ==============================================================================
# 5. fMRI Feature Extraction (Robust NaN Padding version)
# ==============================================================================
def run_fmri_pipeline(df_labels):
    logging.info("="*50)
    logging.info("PHASE 3: fMRI Extraction Pipeline (Robust Padding Edition)")
    logging.info("="*50)
    os.makedirs(FMRI_OUTPUT_DIR, exist_ok=True)
    
    n_regions_total = len(df_labels)
    img_data = nib.load(AAL3_NII_PATH).get_fdata()
    valid_ids = np.unique(img_data.astype(int))
    valid_ids = valid_ids[valid_ids != 0]
    valid_ids.sort()
    
    logging.info(f" Physical ROIs in NIfTI: {len(valid_ids)}, Total ROIs in Atlas TXT: {n_regions_total}.")
    col_indices = [df_labels.index[df_labels['ROI_ID'] == vid].tolist()[0] for vid in valid_ids]
    
    masker = NiftiLabelsMasker(
        labels_img=AAL3_NII_PATH, detrend=True, standardize="zscore_sample",
        standardize_confounds=True, low_pass=0.15, high_pass=0.01, t_r=0.72,
        memory='nilearn_cache', memory_level=1
    )
    
    all_fmri_files = glob.glob(os.path.join(FMRI_INPUT_DIR, '*.nii.gz'))
    
    for fmri_nii_path in all_fmri_files:
        filename = os.path.basename(fmri_nii_path)
        output_xlsx_path = os.path.join(FMRI_OUTPUT_DIR, filename.replace('.nii.gz', '.xlsx'))
        
        if os.path.exists(output_xlsx_path):
            logging.info(f"   [Skipped] {filename}")
            continue
            
        logging.info(f"   [Extracting] {filename} ...")
        try:
            fmri_img = nib.load(fmri_nii_path)
            timeseries_extracted = masker.fit_transform(fmri_img).astype(np.float32)
            
            n_timepoints = timeseries_extracted.shape[0]
            timeseries_padded = np.full((n_timepoints, n_regions_total), np.nan, dtype=np.float32)
            timeseries_padded[:, col_indices] = timeseries_extracted
            
            index = pd.MultiIndex.from_arrays([df_labels['Network'], df_labels['AAL_Label']], names=['Network', 'Region'])
            df_timeseries = pd.DataFrame(timeseries_padded, columns=index, dtype=np.float32)
            df_timeseries.index.name = 'Timepoint'
            
            with pd.ExcelWriter(output_xlsx_path, engine='openpyxl') as writer:
                df_timeseries.to_excel(writer, sheet_name='Timeseries')
                df_labels.drop(columns=['ROI_ID']).to_excel(writer, sheet_name='Metadata', index=False)
                
            logging.info(f"   [Success] {filename} (Successfully padded to {n_regions_total} columns)")
            
        except Exception as e: 
            logging.error(f"     [Error] Extraction failed: {e}")
            
        finally:
            if 'fmri_img' in locals(): del fmri_img
            if 'timeseries_extracted' in locals(): del timeseries_extracted
            if 'timeseries_padded' in locals(): del timeseries_padded
            gc.collect()

# ==============================================================================
# 6. Main Execution Block
# ==============================================================================
if __name__ == "__main__":
    logging.info("Initializing Atlas System & ROI Classifier...")
    
    df_labels = load_aal3_labels(AAL3_TXT_PATH)
    logging.info(f"Atlas Loaded! Total {len(df_labels)} ROIs distributed across {df_labels['Network'].nunique()} macro-networks.")
    
    logging.info("Calculating 3D spatial coordinates (This may take a few seconds)...")
    roi_coords = plotting.find_parcellation_cut_coords(labels_img=AAL3_NII_PATH)
    
    # Run the complete multi-modal pipeline
    run_meg_pipeline(roi_coords, df_labels)
    run_vmd_pipeline()
    run_fmri_pipeline(df_labels)
    
    logging.info("="*50)
    logging.info("ALL DONE! Dual-Modality Extraction & Aperiodic VMD Separation completed.")
    logging.info("="*50)